In [9]:
import math
from dataclasses import dataclass
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import landscape, A3
from reportlab.lib.colors import white, black, HexColor


# =========================================================
# CONFIG
# =========================================================
PDF_FILE = "proper_residential_floorplan.pdf"

PAGE_W, PAGE_H = landscape(A3)

BG = white
C_WALL = black
C_LINE = HexColor("#222222")
C_DIM = HexColor("#666666")
C_FAINT = HexColor("#999999")
C_GUIDE = HexColor("#C8C8C8")

OUTER_WALL = 18
INNER_WALL = 10

PX_PER_M = 75.0

LINE_THIN = 0.8
LINE_MED = 1.0
LINE_DOOR = 1.1
LINE_DIM = 0.7

# building footprint
X0 = 85
Y0 = 85
BW = 980
BH = 620


# =========================================================
# BASIC DRAW HELPERS
# =========================================================
def draw_line(c, x1, y1, x2, y2, lw=1.0, color=C_LINE):
    c.setStrokeColor(color)
    c.setLineWidth(lw)
    c.line(x1, y1, x2, y2)


def draw_rect(c, x, y, w, h, lw=1.0, color=C_LINE, fill=None):
    c.setStrokeColor(color)
    c.setLineWidth(lw)
    if fill is not None:
        c.setFillColor(fill)
        c.rect(x, y, w, h, stroke=1, fill=1)
    else:
        c.rect(x, y, w, h, stroke=1, fill=0)


def fill_rect(c, x, y, w, h, color=C_WALL):
    c.setFillColor(color)
    c.setStrokeColor(color)
    c.rect(x, y, w, h, stroke=0, fill=1)


def draw_text(c, x, y, text, size=10, font="Helvetica", color=C_LINE, center=False):
    c.setFillColor(color)
    c.setFont(font, size)
    if center:
        c.drawCentredString(x, y, text)
    else:
        c.drawString(x, y, text)


def draw_arc(c, cx, cy, r, start_deg, end_deg, lw=1.0, color=C_LINE):
    c.setStrokeColor(color)
    c.setLineWidth(lw)
    c.arc(cx - r, cy - r, cx + r, cy + r, start_deg, end_deg - start_deg)


def area_label(w, h):
    return f"{(w / PX_PER_M) * (h / PX_PER_M):.1f} m²"


# =========================================================
# WALLS
# =========================================================
def wall_h(c, x1, x2, y, thick, gaps=None):
    gaps = sorted(gaps or [])
    cur = x1
    for a, b in gaps:
        if a > cur:
            fill_rect(c, cur, y - thick / 2, a - cur, thick, C_WALL)
        cur = max(cur, b)
    if cur < x2:
        fill_rect(c, cur, y - thick / 2, x2 - cur, thick, C_WALL)


def wall_v(c, x, y1, y2, thick, gaps=None):
    gaps = sorted(gaps or [])
    cur = y1
    for a, b in gaps:
        if a > cur:
            fill_rect(c, x - thick / 2, cur, thick, a - cur, C_WALL)
        cur = max(cur, b)
    if cur < y2:
        fill_rect(c, x - thick / 2, cur, thick, y2 - cur, C_WALL)


# =========================================================
# WINDOWS
# =========================================================
def window_h(c, x, y, w, wall_t):
    inset = wall_t * 0.34
    draw_line(c, x, y - inset, x + w, y - inset, lw=0.9, color=C_LINE)
    draw_line(c, x, y + inset, x + w, y + inset, lw=0.9, color=C_LINE)
    draw_line(c, x, y - inset, x, y + inset, lw=0.7, color=C_LINE)
    draw_line(c, x + w, y - inset, x + w, y + inset, lw=0.7, color=C_LINE)
    draw_line(c, x + w / 2, y - inset, x + w / 2, y + inset, lw=0.5, color=C_FAINT)


def window_v(c, x, y, h, wall_t):
    inset = wall_t * 0.34
    draw_line(c, x - inset, y, x - inset, y + h, lw=0.9, color=C_LINE)
    draw_line(c, x + inset, y, x + inset, y + h, lw=0.9, color=C_LINE)
    draw_line(c, x - inset, y, x + inset, y, lw=0.7, color=C_LINE)
    draw_line(c, x - inset, y + h, x + inset, y + h, lw=0.7, color=C_LINE)
    draw_line(c, x - inset, y + h / 2, x + inset, y + h / 2, lw=0.5, color=C_FAINT)


# =========================================================
# DOORS
# =========================================================
def door_up(c, x, y, w):
    draw_line(c, x, y, x, y + w, lw=LINE_DOOR, color=C_LINE)
    draw_arc(c, x, y, w, 0, 90, lw=0.9, color=C_FAINT)


def door_down(c, x, y, w):
    draw_line(c, x, y, x, y - w, lw=LINE_DOOR, color=C_LINE)
    draw_arc(c, x, y, w, 270, 360, lw=0.9, color=C_FAINT)


def door_left(c, x, y, w):
    draw_line(c, x, y, x - w, y, lw=LINE_DOOR, color=C_LINE)
    draw_arc(c, x, y, w, 180, 270, lw=0.9, color=C_FAINT)


def door_right(c, x, y, w):
    draw_line(c, x, y, x + w, y, lw=LINE_DOOR, color=C_LINE)
    draw_arc(c, x, y, w, 90, 180, lw=0.9, color=C_FAINT)


def double_door_bottom(c, x, y, w):
    half = w / 2
    draw_line(c, x, y, x + half, y, lw=1.0, color=C_LINE)
    draw_line(c, x + w, y, x + half, y, lw=1.0, color=C_LINE)
    draw_arc(c, x, y, half, 0, 90, lw=0.9, color=C_FAINT)
    draw_arc(c, x + w, y, half, 90, 180, lw=0.9, color=C_FAINT)


# =========================================================
# DIMENSIONS
# =========================================================
def dim_h(c, x1, x2, y, off=26, label=None):
    yy = y + off
    draw_line(c, x1, y, x1, yy, lw=LINE_DIM, color=C_DIM)
    draw_line(c, x2, y, x2, yy, lw=LINE_DIM, color=C_DIM)
    draw_line(c, x1, yy, x2, yy, lw=LINE_DIM, color=C_DIM)
    draw_line(c, x1, yy - 4, x1, yy + 4, lw=LINE_DIM, color=C_DIM)
    draw_line(c, x2, yy - 4, x2, yy + 4, lw=LINE_DIM, color=C_DIM)
    draw_text(c, (x1 + x2) / 2, yy + 5, label or f"{(x2-x1)/PX_PER_M:.2f} m", size=8, center=True, color=C_DIM)


def dim_v(c, x, y1, y2, off=26, label=None):
    xx = x - off
    draw_line(c, x, y1, xx, y1, lw=LINE_DIM, color=C_DIM)
    draw_line(c, x, y2, xx, y2, lw=LINE_DIM, color=C_DIM)
    draw_line(c, xx, y1, xx, y2, lw=LINE_DIM, color=C_DIM)
    draw_line(c, xx - 4, y1, xx + 4, y1, lw=LINE_DIM, color=C_DIM)
    draw_line(c, xx - 4, y2, xx + 4, y2, lw=LINE_DIM, color=C_DIM)
    draw_text(c, xx - 3, (y1 + y2) / 2, label or f"{(y2-y1)/PX_PER_M:.2f} m", size=8, color=C_DIM)


# =========================================================
# FURNITURE
# =========================================================
def sofa(c, x, y, w=120, h=52):
    draw_rect(c, x, y, w, h, lw=0.9, color=C_FAINT)
    draw_rect(c, x + 5, y + 5, w - 10, h - 10, lw=0.6, color=C_FAINT)


def table(c, x, y, w=70, h=50):
    draw_rect(c, x, y, w, h, lw=0.8, color=C_FAINT)


def bed(c, x, y, w=95, h=145):
    draw_rect(c, x, y, w, h, lw=0.9, color=C_FAINT)
    draw_rect(c, x + 6, y + h - 22, w - 12, 16, lw=0.6, color=C_FAINT)
    draw_rect(c, x + 6, y + 8, (w - 18) / 2, h - 36, lw=0.5, color=C_FAINT)
    draw_rect(c, x + w / 2 + 3, y + 8, (w - 18) / 2, h - 36, lw=0.5, color=C_FAINT)


def wardrobe(c, x, y, w=72, h=28):
    draw_rect(c, x, y, w, h, lw=0.8, color=C_FAINT)
    draw_line(c, x + w / 2, y, x + w / 2, y + h, lw=0.5, color=C_GUIDE)


def desk(c, x, y, w=90, h=32):
    draw_rect(c, x, y, w, h, lw=0.8, color=C_FAINT)


def kitchen_counter(c, x, y, w, h):
    draw_rect(c, x, y, w, h, lw=0.9, color=C_FAINT)
    step = 28
    if w >= h:
        xx = x + step
        while xx < x + w:
            draw_line(c, xx, y, xx, y + h, lw=0.4, color=C_GUIDE)
            xx += step
    else:
        yy = y + step
        while yy < y + h:
            draw_line(c, x, yy, x + w, yy, lw=0.4, color=C_GUIDE)
            yy += step


def sink(c, x, y, w=28, h=16):
    draw_rect(c, x, y, w, h, lw=0.8, color=C_FAINT)
    c.setStrokeColor(C_FAINT)
    c.setLineWidth(0.8)
    c.circle(x + w / 2, y + h / 2, 3, stroke=1, fill=0)


def wc(c, x, y):
    draw_rect(c, x + 6, y + 18, 20, 12, lw=0.7, color=C_FAINT)
    c.setStrokeColor(C_FAINT)
    c.setLineWidth(0.8)
    c.ellipse(x, y, x + 32, y + 22, stroke=1, fill=0)


def shower(c, x, y, w=68, h=46):
    draw_rect(c, x, y, w, h, lw=0.8, color=C_FAINT)


# =========================================================
# ROOM LABELS
# =========================================================
def room_label(c, name, x, y, w, h):
    draw_text(c, x + w / 2, y + h / 2 + 8, name.upper(), size=11, font="Helvetica-Bold", center=True)
    draw_text(c, x + w / 2, y + h / 2 - 9, area_label(w, h), size=8, center=True, color=C_DIM)


# =========================================================
# TITLE + NORTH ARROW
# =========================================================
def draw_title_block(c):
    tx = PAGE_W - 220
    ty = 22

    draw_rect(c, tx, ty, 170, 66, lw=0.9, color=C_LINE)
    draw_line(c, tx, ty + 22, tx + 170, ty + 22, lw=0.7)
    draw_line(c, tx, ty + 44, tx + 170, ty + 44, lw=0.7)
    draw_line(c, tx + 68, ty, tx + 68, ty + 66, lw=0.7)

    draw_text(c, tx + 8, ty + 50, "PROJECT", size=7)
    draw_text(c, tx + 76, ty + 50, "SYNTHETIC HOUSE", size=7)

    draw_text(c, tx + 8, ty + 28, "TYPE", size=7)
    draw_text(c, tx + 76, ty + 28, "RESIDENTIAL PLAN", size=7)

    draw_text(c, tx + 8, ty + 7, "SHEET", size=7)
    draw_text(c, tx + 76, ty + 7, "A-101", size=7)


def draw_north_arrow(c):
    x = 48
    y = PAGE_H - 75
    draw_line(c, x, y - 28, x, y + 24, lw=1.0)
    path = c.beginPath()
    path.moveTo(x, y + 34)
    path.lineTo(x - 6, y + 20)
    path.lineTo(x + 6, y + 20)
    path.close()
    c.drawPath(path, stroke=1, fill=0)
    draw_text(c, x, y - 42, "N", size=10, font="Helvetica-Bold", center=True)


# =========================================================
# MAIN PLAN
# =========================================================
def generate_floorplan():
    c = canvas.Canvas(PDF_FILE, pagesize=landscape(A3))
    c.setFillColor(BG)
    c.rect(0, 0, PAGE_W, PAGE_H, stroke=0, fill=1)

    # -----------------------------------------------------
    # MAIN GEOMETRY
    # -----------------------------------------------------
    y_corr_top = Y0 + 365
    y_corr_bot = Y0 + 325

    # room guide rectangles
    rooms = {
        "living":   (X0 + 45,  Y0 + 365, 285, 220),
        "dining":   (X0 + 330, Y0 + 365, 180, 220),
        "kitchen":  (X0 + 510, Y0 + 365, 275, 220),
        "master":   (X0 + 45,  Y0 + 160, 245, 165),
        "bed2":     (X0 + 290, Y0 + 160, 220, 165),
        "bath":     (X0 + 510, Y0 + 160, 150, 165),
        "study":    (X0 + 660, Y0 + 160, 125, 165),
        "corridor": (X0 + 45,  Y0 + 325, 740, 40),
    }

    # -----------------------------------------------------
    # OUTER WALLS
    # -----------------------------------------------------
    # top wall with 3 windows
    wall_h(c, X0, X0 + BW, Y0 + BH, OUTER_WALL, gaps=[
        (X0 + 95,  X0 + 175),
        (X0 + 395, X0 + 470),
        (X0 + 620, X0 + 690),
    ])

    # bottom wall with 2 windows + main double door
    main_door_x = X0 + 420
    main_door_w = 110
    wall_h(c, X0, X0 + BW, Y0, OUTER_WALL, gaps=[
        (X0 + 220, X0 + 280),
        (main_door_x, main_door_x + main_door_w),
        (X0 + 590, X0 + 650),
    ])

    # left wall with 2 windows + service door
    wall_v(c, X0, Y0, Y0 + BH, OUTER_WALL, gaps=[
        (Y0 + 240, Y0 + 300),
        (Y0 + 480, Y0 + 535),
        (Y0 + 575, Y0 + 615),   # top-left service entry gap
    ])

    # right wall with 2 windows
    wall_v(c, X0 + BW, Y0, Y0 + BH, OUTER_WALL, gaps=[
        (Y0 + 250, Y0 + 315),
        (Y0 + 470, Y0 + 530),
    ])

    # -----------------------------------------------------
    # INNER WALLS
    # -----------------------------------------------------
    # corridor lines
    wall_h(c, X0 + 85, X0 + 810, y_corr_top, INNER_WALL, gaps=[
        (X0 + 145, X0 + 215),
        (X0 + 395, X0 + 465),
        (X0 + 640, X0 + 705),
    ])
    wall_h(c, X0 + 85, X0 + 810, y_corr_bot, INNER_WALL, gaps=[
        (X0 + 145, X0 + 215),
        (X0 + 395, X0 + 465),
        (X0 + 640, X0 + 705),
        (X0 + 770, X0 + 820),
    ])

    # upper vertical partitions
    wall_v(c, X0 + 330, y_corr_top, Y0 + BH - 42, INNER_WALL)
    wall_v(c, X0 + 510, y_corr_top, Y0 + BH - 42, INNER_WALL)

    # lower vertical partitions
    wall_v(c, X0 + 290, Y0 + 160, y_corr_bot, INNER_WALL)
    wall_v(c, X0 + 510, Y0 + 160, y_corr_bot, INNER_WALL)
    wall_v(c, X0 + 660, Y0 + 160, y_corr_bot, INNER_WALL)
    wall_v(c, X0 + 785, Y0 + 160, y_corr_bot, INNER_WALL)

    # -----------------------------------------------------
    # WINDOWS
    # -----------------------------------------------------
    # outer top
    window_h(c, X0 + 95,  Y0 + BH, 80, OUTER_WALL)
    window_h(c, X0 + 395, Y0 + BH, 75, OUTER_WALL)
    window_h(c, X0 + 620, Y0 + BH, 70, OUTER_WALL)

    # outer bottom
    window_h(c, X0 + 220, Y0, 60, OUTER_WALL)
    window_h(c, X0 + 590, Y0, 60, OUTER_WALL)

    # left side
    window_v(c, X0, Y0 + 240, 60, OUTER_WALL)
    window_v(c, X0, Y0 + 480, 55, OUTER_WALL)

    # right side
    window_v(c, X0 + BW, Y0 + 250, 65, OUTER_WALL)
    window_v(c, X0 + BW, Y0 + 470, 60, OUTER_WALL)

    # interior windows
    window_v(c, X0 + 510, Y0 + 420, 48, INNER_WALL)
    window_v(c, X0 + 660, Y0 + 235, 46, INNER_WALL)
    window_h(c, X0 + 495, y_corr_top, 70, INNER_WALL)

    # -----------------------------------------------------
    # DOORS
    # -----------------------------------------------------
    # top rooms to corridor
    door_up(c, X0 + 145, y_corr_top, 56)
    door_up(c, X0 + 395, y_corr_top, 56)
    door_up(c, X0 + 640, y_corr_top, 56)

    # bottom rooms to corridor
    door_down(c, X0 + 145, y_corr_bot, 52)
    door_down(c, X0 + 395, y_corr_bot, 52)
    door_down(c, X0 + 640, y_corr_bot, 52)
    door_down(c, X0 + 770, y_corr_bot, 44)

    # service entry left
    door_right(c, X0, Y0 + 575, 60)

    # main entrance
    double_door_bottom(c, main_door_x, Y0, main_door_w)

    # -----------------------------------------------------
    # LIGHT GUIDE ROOM BOXES
    # -----------------------------------------------------
    for _, (rx, ry, rw, rh) in rooms.items():
        draw_rect(c, rx, ry, rw, rh, lw=0.35, color=C_GUIDE)

    # -----------------------------------------------------
    # LABELS
    # -----------------------------------------------------
    room_label(c, "Living Room", *rooms["living"])
    room_label(c, "Dining", *rooms["dining"])
    room_label(c, "Kitchen", *rooms["kitchen"])
    room_label(c, "Master Bedroom", *rooms["master"])
    room_label(c, "Bedroom 2", *rooms["bed2"])
    room_label(c, "Bathroom", *rooms["bath"])
    room_label(c, "Study", *rooms["study"])
    draw_text(c, X0 + 455, Y0 + 342, "CORRIDOR", size=10, font="Helvetica-Bold", center=True)

    # -----------------------------------------------------
    # FURNITURE
    # -----------------------------------------------------
    # living
    sofa(c, X0 + 92, Y0 + 475, 132, 48)
    table(c, X0 + 245, Y0 + 472, 50, 42)
    wardrobe(c, X0 + 70, Y0 + 555, 130, 24)

    # dining
    table(c, X0 + 382, Y0 + 445, 86, 58)

    # kitchen
    kitchen_counter(c, X0 + 545, Y0 + 510, 128, 22)
    sink(c, X0 + 590, Y0 + 514, 26, 14)
    kitchen_counter(c, X0 + 690, Y0 + 420, 20, 120)
    table(c, X0 + 665, Y0 + 535, 24, 24)

    # master
    bed(c, X0 + 78, Y0 + 162, 100, 134)
    wardrobe(c, X0 + 215, Y0 + 235, 54, 28)
    table(c, X0 + 185, Y0 + 235, 34, 28)

    # bedroom 2
    bed(c, X0 + 335, Y0 + 162, 90, 128)
    wardrobe(c, X0 + 455, Y0 + 236, 48, 28)
    table(c, X0 + 430, Y0 + 236, 30, 28)

    # bathroom
    wc(c, X0 + 548, Y0 + 248)
    sink(c, X0 + 585, Y0 + 248, 34, 18)
    shower(c, X0 + 540, Y0 + 180, 82, 46)

    # study
    table(c, X0 + 700, Y0 + 230, 82, 42)
    desk(c, X0 + 712, Y0 + 195, 92, 28)

    # -----------------------------------------------------
    # DIMENSIONS
    # -----------------------------------------------------
    dim_h(c, X0, X0 + BW, Y0 + BH, off=34, label=f"{BW / PX_PER_M:.2f} m")
    dim_v(c, X0, Y0, Y0 + BH, off=28, label=f"{BH / PX_PER_M:.2f} m")
    dim_h(c, X0 + 45, X0 + 330, Y0 + 565, off=-12, label="Living width")
    dim_v(c, X0 + 45, Y0 + 365, Y0 + 585, off=-14, label="Living depth")

    # -----------------------------------------------------
    # TITLE / NORTH ARROW
    # -----------------------------------------------------
    draw_title_block(c)
    draw_north_arrow(c)

    c.showPage()
    c.save()
    print(f"Saved: {PDF_FILE}")


if __name__ == "__main__":
    generate_floorplan()

Saved: proper_residential_floorplan.pdf
